# Claims ETL Pipeline (Multi-DB Fan-Out)   

"Clean code" refers to code that is easy to read, understand, maintain, and extend. It follows best practices, has clear intent, and is written in a way that minimizes confusion for future developers (including your future self). I focused on readability, simplicity, and documentation. Please let owner know if you see any opportunities for improving the cleanliness of this code. Particularly in regards to testability, modularity, and error handling.

- urlib.parse.quote_plus  
- pandas  
- sqlalchemy.create_engine  
- sqlalchemy.text  

There is no need to reinstall packages once they are installed so once you run it, please comment out by selecting and clicking CTRL + /.

In [ ]:
%%capture

# !python -m pip install cx_Oracle --upgrade
!python -m pip install oracledb --upgrade --user
!python -m pip install openpyxl --upgrade --user
!python -m pip install pandasql --upgrade --user
!python -m pip install 'pyarrow>=10.0.1,<10.1.0' --user
!python -m pip install pandoc --upgrade  --user
!python -m pip install datetime --upgrade  --user

In [ ]:
# from cx_Oracle import DatabaseError
from oracledb import DatabaseError
from urllib.parse import quote_plus
import pandas as pd
from sqlalchemy import create_engine, text
from datetime import datetime

The remaining code is to run 2 claims queries against 8 databases each and four (4) more claims queries against one database each. 

Create fxn vault_auth() to create dict for user and pwd.

In [ ]:
%run vault_auth.py

Define function oracle_engine that:
    - takes 'dest' as a string
    - calls vault_auth
    - gets service_name from parameter
    - assigns hostname
    - defines connection_link
    - assigns engine

A SQLAlchemy Engine references both a Dialect and a Pool, which together interpret the DBAPI’s module functions as well as the behavior of the database.

The engine below creates a Dialect object tailored towards Oracle, and a Pool object which will establish a DBAPI connection at hostname:1521 when a connection request is first received.  
  
Note that the Engine and its underlying Pool do not establish the first actual DBAPI connection until the Engine.begin() method is called in get_readsql() below.

In [ ]:
def oracle_engine(dest: str):
    creds_dict = vault_auth()

    service_name = dest
    hostname = "oracle-host.example.com"

#     connection_link = 'oracle+cx_oracle://'+ creds_dict["db_username"] + ':' + creds_dict["db_password"] + \
#                        '@' + hostname + ':1521/?service_name=' + \
#                        service_name
    connection_link = 'oracle+oracledb://'+ creds_dict["db_username"] + ':' + creds_dict["db_password"] + \
                       '@' + hostname + ':1521/?service_name=' + \
                       service_name
    engine = create_engine(connection_link, echo=False)
    return engine

Define get_readsql  
Params: 
- sql_query: str 
- dest: str  
  
Summary: Execute an SQL query on an Oracle database and return the results as a Pandas DataFrame
line 3: call oracle_engine to create connection engine

line 5: `with` keyword starts a transaction context using the connection engine while ensuring that the connection will be closed when the block is exited, even if an error occurs.  
Inside the transaction context or `with` block:  
`sql=text(sql_query)`: Wraps the SQL query in a SQLAlchemy text object to ensure it's treated as a raw SQL string.  
`pandas.read_sql_query()` function is used to read the results of a SQL query into a pandas DataFrame.  
  `con`: The connection object to the database. This can be a SQLAlchemy engine or a raw database connection.

In [ ]:
def get_readsql(sql_query: str, dest: str):

    engine = oracle_engine(dest)

    with engine.begin() as conn:
        df = pd.read_sql_query(sql=text(sql_query), con=conn)
    engine.dispose()

    return df

# Save Queries to Strings
### Queries
1. California Commercial (Commercial)
1. Marketplace Stratum 5 (db_marketplace@example.com)
1. Medicaid Stratum 4
1. Medicaid Stratum 5
1. Medicare Stratum 5
1. paid over 125% over billed (db_region_8@example.com)
1. All States Marketplace Paid Over Billed (db_marketplace@example.com)

In [ ]:
%run queries/commercial_prepay_pob_hd.py
%run queries/marketplace_t5.py
%run queries/medicare_t5.py
%run queries/medicaid_t5.py
%run queries/medicaid_t4.py
%run queries/state_prepay.py
%run queries/all_marketplace_pob.py
%run qry_hd.py
%run qry_prepay.py
%run qry_pob.py

## Create list of databases, `db_list`.

In [ ]:
#databases medicaid queries are run against
region_dbs = [
    'db_region_1',
    'db_region_2',
    'db_region_3',
    'db_region_4',
    'db_region_5',  
    'db_region_6',
    'db_region_7',
    'db_region_8'
]

# list of databases with each listed the number of times it is run
db_list = region_dbs * 2 + ['db_medicare' , 'db_marketplace' , 'db_commercial', 'db_region_8', 'db_marketplace']
print("Number of Items in db_list: ", len(db_list))
print("List: ", db_list)


# Create list of Queries, `query_list`.

In [ ]:

query_list = [qry_medicaid_t5] * len(region_dbs) + [qry_medicaid_t4] * len(region_dbs) + [qry_medicare_t5, qry_marketplace_t5, qry_commercial, qry_state_prepay, qry_all_marketplace_pob] # region_5 change 8 to 7
print("Number of Queries: ",len(query_list))
print("Type of Items in List: ", type(query_list[0]))

### Create a list of lists of two item tuples grouped by query. Each tuple contains a database and a query. 
First create a list of two items tuples.  
Use a list comprehension to create a list containing a list of the first 8 tuples and a list of the second 8 tuples.  
Use a loop to add each of the remaining tuples as single item lists.  

In [ ]:
zip_dest_query_orig = list(zip(db_list, query_list))
zip_dest_query = [zip_dest_query_orig[i:i + len(region_dbs)] for i in [0, len(region_dbs)]]        
for i in range(len(region_dbs) * 2,  len(query_list)): zip_dest_query.append([zip_dest_query_orig[i]]) 

#### print the database of each of the 21 queries 

In [ ]:
# QUALITY CHECK
# print the database of each of the 21 queries 
for lst in range(len(zip_dest_query)):
    for qry in range(len(zip_dest_query[lst])):
        print(zip_dest_query[lst][qry][0])

QUALITY CHECK

In [ ]:
print("len(zip_dest_query_orig):", len(zip_dest_query_orig))
print("type(zip_dest_query_orig[0]):", type(zip_dest_query_orig[0]))
print("len(zip_dest_query_orig[0]):", len(zip_dest_query_orig[0]))

print("new length of outer list" , len(zip_dest_query)) 

for lst in range(len(zip_dest_query)):
    print(f"length of list {lst + 1} ", len(zip_dest_query[lst]))

QUALITY CHECK: A list of lists of tuples.
Each tuple contains a a db in position 0 and query in position 1
[[(),(),...],[(),(),...],[(),(),...],...]

In [ ]:
print("First element of first/only tuple in fifth list: zip_dest_query[4][0][0]:")
print(zip_dest_query[4][0][0])
print()
print("Second element of first/only tuple in fifth list: zip_dest_query[4][0][1]:")
print()
print(zip_dest_query[4][0][1])

## Define Excel Export Parameters

Excel spreadsheets are exported to the users JupyterHub home directory.

| Database      | Query    | Spreadsheet |
|-------------|----------|---------------
| db_region_1 | qry_medicaid_t5 | Tier5-Medicaid.xlsx |
| db_region_2 | | |
| db_region_3 | | |
| db_region_4 | | |
| db_region_5 | | |
| db_region_6 | | |
| db_region_7 | | |
| db_region_8 | | |
| db_region_1 | qry_medicaid_t4 | Tier4-Medicaid.xlsx  |
| db_region_2 | | |
| db_region_3 | | |
| db_region_4 | | |
| db_region_5 | | |
| db_region_6 | | |
| db_region_7 | | |
| db_region_8 | | |
| db_medicare | McrStratum5 |  Tier5- Medicare.xlsx  |
| db_marketplace | MkpStratum5 | Tier5-Marketplace.xlsx | 
| db_commercial | qry_commercial-PrepayPobHd | CA-COM Nightly Pull.xlsx | 
| db_region_8 | Prepay-All | FL_Medicaid_Prepay.xlsx |
| db_marketplace | qry_all_marketplace_pob | all_marketplace_pob.xlsx | 

In [ ]:
# Format datetime string to exclude seconds and milliseconds
# now = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
# now = datetime.datetime.now().strftime('_%m%d%Y_%H%M')
now = datetime.now().strftime('%m%d%Y_%H%M')

writer_list = [
      'claims_medicaid_t5_'  + now + '.xlsx'
    , 'claims_medicaid_t4_'  + now + '.xlsx'
    , 'claims_medicare_t5_'  + now + '.xlsx'    
    , 'claims_marketplace_t5_'  + now + '.xlsx'
    , 'claims_commercial_'  + now + '.xlsx'    
    , 'claims_state_prepay_'  + now + '.xlsx'
    , 'claims_marketplace_pob_'  + now + '.xlsx'
]

hd_commercial = [
    'PLATFORM',
    'Audit Type',
    'Claim Number',
    'Processing ID',
    'Processed Date',
    'Health Plan',
    'STATE',
    'PRODUCT',
    'Billed Amount',
    'Allowed Amount',
    'Paid Amount',
    'Provider Name',
    'Provider Type',
    'Provider Specialty',
    'Tax ID',
    'Par or Non-Par',
    'Bill Type',
    'Place of Service',
    'Affiliation Number',
    'NOTES',
    'Penalty Amount'
]

hd_qry_all_marketplace_pob = [
  'Platform',
 'Audit Type',
 'Audit Sub Type',
 'Claim Number',
 'Processing ID',
 'Processed Date',
 'Health Plan',
 'state',
 'Product',
 'Billed Amount',
 'Allowed Amount',
 'Paid Amount',
 'Provider Name',
  'Provider Type',
 'Provider Specialty',
 'Tax ID',
 'Par or Non-Par',
  'Bill Type',
 'Claim Status',
 'Place of Service',
 'Provider Number',
  'Affiliation Number']

hd_medicaid_t5 = hd_medicaid_t4 = hd_medicare_t5 = hd_marketplace_t5 = hd_qry_state_prepay = [
    'PLATFORM',
    'Audit Type',
    'Audit Sub Type',
    'ClaimNumber',
    'Processing ID',
    'Processed Date',
    'Health Plan',
    'STATE',
    'PRODUCT',
     'Billed Amount',
     'Allowed Amount',
     'Paid Amount',
    'Provider Name',
    'Provider Type',
    'Provider Specialty',
    'Tax ID',
    'Par or Non-Par',
    'Bill Type',
    'Place of Service',
    'Affiliation Number',
    'NOTES',
    'Interest Amount',
    'Penalty Amount'
    ]



header_nlist = [
    hd_medicaid_t5
    ,hd_medicaid_t4
    ,hd_medicare_t5
    ,hd_marketplace_t5
    ,hd_commercial    
    ,hd_qry_state_prepay
    ,hd_qry_all_marketplace_pob
]

Check results by printing the item count of the `zip_dest_query` list object, and the item count for its first and third inner lists. 

In [ ]:
# QUALITY CHECK
print(len(zip_dest_query))
print(len(zip_dest_query[0]))
print(len(zip_dest_query[2]))

In [ ]:
# QUALITY CHECK
print(type(zip_dest_query[2][0]))


### Define parameters and wrapper functions write_to_excel and run_query.
`write_to_excel` will take the dataframe, the export file name, and a list of headers and export Excel files to the user's JupyterHub home directory. It announces its start and end.

`run_query` has the drawback that is requires an object is created before its execution, the `global_ex_data` list object. It will take a two-item tuple object including a database name in position 0 and a query in position 1, run the query on the database and save the results to a dataframe, then append the dataframe to a list of query results. 

This function will print "Running Query: followed by the database name when it begins and print "Query Completed:" with the database name and "Rows: " with the number of rows in the output. 

If it fails it will print "Query Failed:" followed by the database name, but it will continue to the next step in the process.

In [ ]:
import datetime
import logging
from urllib.parse import quote_plus

    
def run_query(query):
    
    try:
        
        print("Running Query: ",query[0] )
        start_time = datetime.datetime.now()

        # Configure logging
        logging.basicConfig(filename = f"Audit_{writer_list[i]}.log", level=logging.INFO)

        temp_ex_df = get_readsql(dest = query[0], sql_query = query[1])

        global_ex_data.append(temp_ex_df)

        # Log the execution
        print()
        print("Query Completed: ", query[0] , "Rows: ", len(temp_ex_df))
        logging.info('Script executed successfully')
        end_time = datetime.datetime.now()
        diff = end_time - start_time
        print("Query processing (min): ", round((diff.days * 24 * 60) + (diff.seconds/60), 2))
        print("Query End Time: ", now, " CT")


#     except DatabaseError as e:
#     except cx_Oracle.DatabaseError as e:
    except OperationalError as e:
        error_obj, = e.args
        print()
        print(f"Error Message: {query[0]} - Error: {error_obj.message}")
        logging.error(f"Error Message: {query[0]} - Error: {error_obj.message}")
        pass
    except InternalError as e:
        error_obj, = e.args
        print()
        print(f"Error Message: {query[0]} - Error: {error_obj.message}")
        logging.error(f"Error Message: {query[0]} - Error: {error_obj.message}")
        pass
    except ProgrammingError as e:
        error_obj, = e.args
        print()
        print(f"Error Message: {query[0]} - Error: {error_obj.message}")
        logging.error(f"Error Message: {query[0]} - Error: {error_obj.message}")
        pass
    except NotSupportedError as e:
        error_obj, = e.args
        print()
        print(f"Error Message: {query[0]} - Error: {error_obj.message}")
        logging.error(f"Error Message: {query[0]} - Error: {error_obj.message}")
        pass
    except DatabaseError as e:
        error_obj, = e.args
        print()
        print(f"Error Message: {query[0]} - Error: {error_obj.message}")
        logging.error(f"Error Message: {query[0]} - Error: {error_obj.message}")
        pass
    except Exception as e:
        print()
        print(f"Unexpected Error for query {query[0]}: {e}")
        logging.error(f"Unexpected Error for query {query[0]}: {e}")
        pass

def write_to_excel(df, excel_writer, header):
    try:
        print('Writing ' + excel_writer + ' beginning:')

        df.to_excel(
            excel_writer=excel_writer
            ,sheet_name='Bulk_Intake'
            ,float_format="%.2f"
            ,header=header
            ,index=False
            ,engine='openpyxl'
        )

        print('Writing ' + excel_writer + ' finished')
        print()
        
    except Exception as e:
        print(f"An error occurred while saving the Excel file: {e}")
        print()
        # continue
        pass


### Parallel Processing

multiprocessing.pool.Pool

Line 1 the ThreadPool library of the pool class of the multiprocessing module is imported for use and called Pool
Line 3: A ThreadPool object with 8 processes is created.
Line 5 loop:
    loops through the `zip_dest_query` list.
    Each time is runs the query, database pairs in the inner list and outputs the results to excel.
lambda_fxn = lambda(arguments:expression) 

<a href="querySelectedClaimsMultiProcessing.Pool.ThreadPoolLogic.ipynb" target="_blank">Complete Explanation</a>

### Run all 20 queries
#### The results of the first and second eight are saved in one spreadsheet each, producing two spreadsheets.
#### The results of the following 5 queries are saved into on spreadsheet each, producing 5 spreadsheets.

df names not used in write_to_excel because we must change the headers to match the desired output


In [ ]:
# %%capture output

from multiprocessing.pool import ThreadPool as Pool

pool = Pool(processes = 8)
df = pd.DataFrame()
# Create datasets with dynamic names in a dictionary, df_dict
df_dict = {}
### run_query logic:
# temp_ex_df = get_readsql(dest = query[0], sql_query = query[1])
# global_ex_data.append(temp_ex_df)

for i in range(len(zip_dest_query)): #0,1,2,3,4,5,6
    print()
    print('Beginning iteration, ' + str(i))
    df_name = f"df_{zip_dest_query[i][0][0]}_{i}"
    
    global_ex_data = [] # old
    start_time = datetime.datetime.now()
    
    # Map queries in the current batch
    pool.map(lambda each_query: run_query(each_query), zip_dest_query[i]) # 8, 8, 1,1,1,1 
    
    if zip_dest_query[i][0][0] == 'db_commercial':  # Commercial: dont print to excel
        if global_ex_data:
            df = pd.concat(global_ex_data)
            df.loc[:,'Processed Date'] = df['Processed Date'].str.strip()
            df.loc[:,'Processed Date'] = df['Processed Date'].astype(int)
            df.loc[:,'Processed Date'] = df['Processed Date'].astype(str)
            print(f"Commercial branch is not written to Excel.")
            print("Rows total: ", len(df))
#             print(output)
        else:
            print(f"No data to write for iteration {i}.")
#             print(output)
            # Concatenate results if any queries succeeded
    else: 

        if global_ex_data:
            df_dict[df_name] = pd.concat(global_ex_data)
    #         df_dict[df_name]['Processed Date'] = pd.to_datetime(df_dict[df_name]['Processed Date'], format='%m/%d/%Y')
            df_dict[df_name].loc[:,'Processed Date'] = df_dict[df_name]['Processed Date'].str.strip()
            df_dict[df_name].loc[:,'Processed Date'] = df_dict[df_name]['Processed Date'].astype(int)
            df_dict[df_name].loc[:,'Processed Date'] = df_dict[df_name]['Processed Date'].astype(str)
            print("Rows total: ", len(df_dict[df_name]))
            write_to_excel(df_dict[df_name], writer_list[i], header_nlist[i])
    #             print(output)
        else:
            print(f"No data to write for iteration {i}.")
    #             print(output)
    
    # Measure and log the time taken for the iteration
    end_time = datetime.datetime.now()
    diff = end_time - start_time
    end = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    print("Printed: ", end)
    print("Process and Print (min): ", round((diff.days * 24 * 60) + (diff.seconds/60), 2))
#     print(output)

# Save output to a file
# with open('output.txt', 'w') as f:
#     f.write(output.stdout)

# print(output): Print output to the output cell


#### For Manual Process in case there is an error with one of the runs above.

In [ ]:
# #### DELETE ME ##########
# ########################

# from multiprocessing.pool import ThreadPool as Pool

# pool = Pool(processes = 8)
# df = pd.DataFrame()

# # for i in range(len(zip_dest_query)): #0,1,2,3,4,5,6
# for i in (4,): #0,1,2,3,4,5,6
#     print()
#     print('Beginning iteration, ' + str(i))

#     global_ex_data = [] # old
#     start_time = datetime.datetime.now()
    
#     # Map queries in the current batch
#     pool.map(lambda each_query: run_query(each_query), zip_dest_query[i]) # 8, 8, 1,1,1,1 
    
#     if zip_dest_query[i][0][0] == 'db_commercial':  # Commercial: dont print to excel
#         if global_ex_data:
#             df = pd.concat(global_ex_data)
# #             amounts = ['BILLED AMOUNT', 'ALLOWED AMOUNT', 'PAID AMOUNT']
#             amounts = ['Billed Amount','Allowed Amount', 'Paid Amount']
            
#             # Remove dollar signs and commas, then convert to float64
#             for col in amounts:
#                 df[col] = df[col].replace({'\$': '', ',': ''}, regex=True).astype('float64')

#             # Convert all, but amounts to string 
#             df = df.astype({col: 'string' for col in df.columns if col not in amounts})
#             print()
#             print("Rows total: ", len(global_ex_df))
#         else:
#             print(f"No data to write for iteration {i}.")
#             # Concatenate results if any queries succeeded
#     else: 
#         if global_ex_data:
#             global_ex_df = pd.concat(global_ex_data)
#             print("Rows total: ", len(global_ex_df))
#             write_to_excel(global_ex_df, writer_list[i], header_nlist[i])
#         # write_to_excel(global_ex_df, writer_list[i], global_ex_df.columns.tolist())
#         else:
#             print(f"No data to write for iteration {i}.")
    
#     # Measure and log the time taken for the iteration
#     end_time = datetime.datetime.now()
#     diff = end_time - start_time
#     print(end_time)
#     print("Process and Print (min): ", round((diff.days * 24 * 60) + (diff.seconds/60), 2))


### In each of the next 3 cells 
1. Run a sql query on df from commercial qry.
1. Return the results into a dataframe.
1. Export each dataframe to an Excel file.

The use of pandasql.sqldf was replaced with the object-oriented approach of using and reusing an instance of PandaSQL class, which is more efficient.

In [ ]:
from pandasql import PandaSQL

# Initialize PandaSQL
ps = PandaSQL()

comm_hd = ps(qry_hd)
write_to_excel(comm_hd, 'claims_commercial_hd_' + now + '.xlsx', comm_hd.columns.tolist())

comm_pob = ps(qry_pob)
write_to_excel(comm_pob, 'claims_commercial_pob_' + now + '.xlsx', comm_pob.columns.tolist())

comm_prepay = ps(qry_prepay)
write_to_excel(comm_prepay, 'claims_commercial_prepay_' + now + '.xlsx', comm_prepay.columns.tolist())

In [ ]:
excel_files = (

      'claims_medicaid_t5_' + now + '.xlsx'
    , 'claims_medicaid_t4_' + now + '.xlsx'
    , 'claims_medicare_t5_' + now + '.xlsx'  
    
    , 'claims_marketplace_t5_' + now + '.xlsx'
    , 'claims_commercial_prepay_' + now + '.xlsx'
    , 'claims_commercial_hd_' + now + '.xlsx'
    
    , 'claims_commercial_pob_' + now + '.xlsx'
    , 'claims_state_prepay_' + now + '.xlsx'
    , 'claims_marketplace_pob_' + now + '.xlsx'
)

# %run cwsFormatting.ipynb

In [ ]:
# !python -m pip install mailer --upgrade
from mailer import sendEmail 

for xlsx in excel_files:
    sendEmail(
        sender = "sender@example.com",
        to = ['sender@example.com', 
              'team-alias@example.com',
              "recipient1@example.com", 
              "recipient2@example.com", 
              "recipient3@example.com", 
              "recipient4@example.com",
              "recipient5@example.com", 
              "recipient6@example.com",
              'recipient7@example.com',
              'recipient8@example.com',
              'recipient9@example.com'
             ],
    title = 'Quality Audit Platform - Claims',
        msg = 'Bulk_Intake',
        attachment_path = xlsx 
    )

### For manual processing in case of an error effecting one email

In [ ]:
# from mailer import sendEmail 

# from datetime import datetime
# Format datetime to exclude seconds and milliseconds
# now = datetime.now().strftime('%Y-%m-%d %H:%M')


# output = (
# now + '-' + 'claims_t5_medicaid.xlsx',    
#     now + '-' + 'claims_t4_medicaid.xlsx',
# now + '-' + 'claims_t5_medicare.xlsx'#,    
#     now + '-' + 'claims_t5_marketplace.xlsx',
#     now + '-' + 'claims_commercial_prepay.xlsx',
#     now + '-' + 'claims_commercial_hd.xlsx',
#     now + '-' + 'claims_commercial_pob.xlsx',
#     now + '-' + 'claims_state_medicaid_prepay.xlsx',
#     now + '-' + 'all_marketplace_pob.xlsx'
# )
# for xlsx in output:
#     sendEmail(
#         sender = "sender@example.com",
#         to = 'team-alias@example.com',
#['recipient4@example.com', "sender@example.com"],
#         title = 'Quality Audit Platform - Claims',
#         msg = 'Bulk_Intake',
#         attachment_path = xlsx 
#     )